# 02 — Pandas Fundamentals

**Topics covered:**
1. Series & DataFrame basics
2. Loading & inspecting data
3. Selecting & filtering
4. Cleaning: missing values & duplicates
5. Adding & transforming columns
6. GroupBy & aggregation
7. Merging DataFrames
8. Time series
9. Mini-exercises

In [39]:
import numpy as np
import pandas as pd
from sklearn import datasets

def load_iris() -> pd.DataFrame:
    """Iris flower dataset (150 rows, 4 features + target)."""
    data = datasets.load_iris(as_frame=True)
    df = data.frame.copy()
    df["species"] = df["target"].map(dict(enumerate(data.target_names)))
    return df


def load_wine() -> pd.DataFrame:
    """Wine recognition dataset (178 rows, 13 features + target)."""
    data = datasets.load_wine(as_frame=True)
    df = data.frame.copy()
    df["class"] = df["target"].map(dict(enumerate(data.target_names)))
    return df


def load_diabetes() -> pd.DataFrame:
    """Diabetes progression dataset (442 rows, 10 features + target)."""
    data = datasets.load_diabetes(as_frame=True)
    return data.frame.copy()


def make_timeseries(n: int = 365, freq: str = "D", seed: int = 42) -> pd.DataFrame:
    """Generate a synthetic daily time-series with trend + seasonality + noise."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2023-01-01", periods=n, freq=freq)
    trend = np.linspace(0, 10, n)
    seasonal = 5 * np.sin(2 * np.pi * np.arange(n) / 30)
    noise = rng.normal(0, 1, n)
    return pd.DataFrame({"date": dates, "value": trend + seasonal + noise})


def make_sales(seed: int = 42) -> pd.DataFrame:
    """Generate a synthetic sales DataFrame for practice."""
    rng = np.random.default_rng(seed)
    regions = ["North", "South", "East", "West"]
    products = ["Widget A", "Widget B", "Gadget X"]
    rows = []
    for month in range(1, 13):
        for region in regions:
            for product in products:
                rows.append({
                    "month": month,
                    "region": region,
                    "product": product,
                    "units_sold": int(rng.integers(50, 300)),
                    "unit_price": round(float(rng.uniform(9.99, 49.99)), 2),
                })
    df = pd.DataFrame(rows)
    df["revenue"] = df["units_sold"] * df["unit_price"]
    return df

In [36]:
import sys, os
import numpy as np
import pandas as pd

import os
print(os.getcwd())
## from ds_sandbox.dataset import load_iris, make_sales, make_timeseries
print(f'pandas version: {pd.__version__}')
# print(f'Project root: {project_root}')  # verify the path

/
pandas version: 2.1.4


## 1. Series & DataFrame Basics

In [24]:
# Series
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='scores')
print(s)
print('\nMean:', s.mean(), '| Max:', s.max())

a    10
b    20
c    30
d    40
Name: scores, dtype: int64

Mean: 25.0 | Max: 40


In [25]:
# DataFrame from dict
df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol', 'Dave'],
    'age':   [25, 30, 35, 28],
    'score': [88.5, 92.0, 79.3, 95.1],
    'passed': [True, True, False, True],
})
df

,name,age,score,passed
0,Alice,25,88.5,True
1,Bob,30,92.0,True
2,Carol,35,79.3,False
3,Dave,28,95.1,True


## 2. Loading & Inspecting Data

In [40]:
iris = load_iris()

print('Shape       :', iris.shape)
print('Columns     :', iris.columns.tolist())
print('\nData types:\n', iris.dtypes)
print('\nMissing values:\n', iris.isnull().sum())
iris.head()

Shape       : (150, 6)
Columns     : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target', 'species']

Data types:
 sepal length (cm)    float64
sepal width (cm)     float64
petal length (cm)    float64
petal width (cm)     float64
target                 int64
species               object
dtype: object

Missing values:
 sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
target               0
species              0
dtype: int64


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [41]:
iris.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


## 3. Selecting & Filtering

In [42]:
# Column selection
print(iris[['sepal length (cm)', 'species']].head())

# Boolean filter
setosa = iris[iris['species'] == 'setosa']
print(f'\nSetosa rows: {len(setosa)}')

# .loc (label) and .iloc (position)
print('\n.loc [0:2, name+species]:')
print(iris.loc[0:2, ['sepal length (cm)', 'species']])

print('\n.iloc [0:3, 0:2]:')
print(iris.iloc[0:3, 0:2])

   sepal length (cm) species
0                5.1  setosa
1                4.9  setosa
2                4.7  setosa
3                4.6  setosa
4                5.0  setosa

Setosa rows: 50

.loc [0:2, name+species]:
   sepal length (cm) species
0                5.1  setosa
1                4.9  setosa
2                4.7  setosa

.iloc [0:3, 0:2]:
   sepal length (cm)  sepal width (cm)
0                5.1               3.5
1                4.9               3.0
2                4.7               3.2


## 4. Cleaning: Missing Values & Duplicates

In [43]:
# Inject some NaN values for practice
dirty = iris.copy()
rng = np.random.default_rng(0)
idx = rng.choice(len(dirty), size=10, replace=False)
dirty.loc[idx, 'sepal length (cm)'] = np.nan

print('Missing before:', dirty.isnull().sum().sum())

# Option A: drop rows with any NaN
dropped = dirty.dropna()
print('Rows after dropna:', len(dropped))

# Option B: fill with column mean
filled = dirty.fillna(dirty.mean(numeric_only=True))
print('Missing after fillna:', filled.isnull().sum().sum())

# Duplicates
print('Duplicate rows:', iris.duplicated().sum())

Missing before: 10
Rows after dropna: 140
Missing after fillna: 0
Duplicate rows: 1


## 5. Adding & Transforming Columns

In [44]:
df = iris.copy()

# New column from arithmetic
df['sepal_area'] = df['sepal length (cm)'] * df['sepal width (cm)']

# Apply / map
df['length_category'] = pd.cut(
    df['sepal length (cm)'], bins=3, labels=['small', 'medium', 'large']
)

# Rename columns (clean up spaces)
df = df.rename(columns=lambda c: c.replace(' ', '_').replace('(cm)', '').strip('_'))

df[['sepal_length', 'sepal_area', 'length_category', 'species']].head(10)

,sepal_length,sepal_area,length_category,species
0,5.1,17.85,small,setosa
1,4.9,14.70,small,setosa
2,4.7,15.04,small,setosa
3,4.6,14.26,small,setosa
4,5.0,18.00,small,setosa
5,5.4,21.06,small,setosa
6,4.6,15.64,small,setosa
7,5.0,17.00,small,setosa
8,4.4,12.76,small,setosa
9,4.9,15.19,small,setosa


## 6. GroupBy & Aggregation

In [45]:
sales = make_sales()

# Revenue by region
region_rev = sales.groupby('region')['revenue'].sum().sort_values(ascending=False)
print('Revenue by region:\n', region_rev)

# Multi-level aggregation
summary = sales.groupby(['region', 'product']).agg(
    total_units=('units_sold', 'sum'),
    total_revenue=('revenue', 'sum'),
    avg_price=('unit_price', 'mean'),
).round(2)

print('\nSummary table:')
summary.head(8)

Revenue by region:
 region
South    200811.49
West     195474.95
East     181362.26
North    181308.59
Name: revenue, dtype: float64

Summary table:


total_units  total_revenue  avg_price
region product                                        
East   Gadget X         1766       58301.95      30.46
       Widget A         1770       52622.06      30.27
       Widget B         2139       70438.25      32.60
North  Gadget X         2005       58144.74      27.51
       Widget A         2216       64679.49      29.37
       Widget B         2010       58484.36      28.49
South  Gadget X         2125       60042.38      28.10
       Widget A         2074       68714.52      34.15

In [46]:
# Pivot table
pivot = sales.pivot_table(
    values='revenue', index='product', columns='region', aggfunc='sum'
).round(0)
pivot

region,East,North,South,West
product,,,,
Gadget X,58302.0,58145.0,60042.0,67154.0
Widget A,52622.0,64679.0,68715.0,46563.0
Widget B,70438.0,58484.0,72055.0,81757.0


## 7. Merging DataFrames

In [47]:
products = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Gadget X', 'Gadget Y'],
    'category': ['Widget', 'Widget', 'Gadget', 'Gadget'],
    'weight_kg': [0.5, 0.8, 1.2, 2.0],
})

# Inner join (only matching keys)
merged = pd.merge(sales, products, on='product', how='left')
print('Merged shape:', merged.shape)
merged[['product', 'category', 'revenue', 'weight_kg']].head()

Merged shape: (144, 8)


,product,category,revenue,weight_kg
0,Widget A,Widget,1983.60,0.5
1,Widget B,Widget,10772.19,0.8
2,Gadget X,Gadget,976.96,1.2
3,Widget A,Widget,10978.24,0.5
4,Widget B,Widget,9653.19,0.8


## 8. Time Series

In [49]:
ts = make_timeseries()
ts = ts.set_index('date')

print('Date range:', ts.index.min(), '->', ts.index.max())

# Resample to monthly mean
monthly = ts.resample('M').mean()
print('\nMonthly mean (first 6):')
print(monthly.head(6))

# Rolling 7-day average
ts['rolling_7'] = ts['value'].rolling(7).mean()
ts.head(10)

Date range: 2023-01-01 00:00:00 -> 2023-12-31 00:00:00

Monthly mean (first 6):
               value
date                
2023-01-31  0.497445
2023-02-28  1.270468
2023-03-31  1.932349
2023-04-30  2.600296
2023-05-31  3.692700
2023-06-30  4.507881


,value,rolling_7
date,,
2023-01-01,0.304717,NaN
2023-01-02,0.027047,NaN
2023-01-03,2.839079,NaN
2023-01-04,3.961909,NaN
2023-01-05,1.874579,NaN
2023-01-06,3.165310,NaN
2023-01-07,5.047958,2.460086
2023-01-08,4.848675,3.109222
2023-01-09,5.175589,3.844728


## 9. Mini-Exercises

In [52]:
# Exercise 1: From the iris dataset, find the species with the highest mean petal length
# Your code here
# Find species with highest mean petal length
mean_petal = df.groupby('species')['petal_length'].mean()
best_species = mean_petal.idxmax()
mean_petal


species
setosa        1.462
versicolor    4.260
virginica     5.552
Name: petal_length, dtype: float64

In [66]:
# Exercise 2: From the sales dataset, find the top 3 (region, product) combos by revenue
# Your code here
sales = make_sales()
sales.groupby(["region","product"]).agg(
	total_revenue=('revenue', 'sum')
).nlargest(3, "total_revenue")


,,total_revenue
region,product,
West,Widget B,81757.46
South,Widget B,72054.59
East,Widget B,70438.25


In [70]:
# Exercise 3: Compute a 30-day rolling standard deviation on the timeseries
# Your code here
# Load and prepare timeseries
ts = make_timeseries()
ts = ts.set_index('date')
print("Dataset shape:", ts.shape)
print(ts.head())
print("\nColumns:", ts.columns.tolist())

# Compute 30-day rolling standard deviation
ts['rolling_30_std'] = ts['value'].rolling(window=30).std()
ts

Dataset shape: (365, 1)
               value
date                
2023-01-01  0.304717
2023-01-02  0.027047
2023-01-03  2.839079
2023-01-04  3.961909
2023-01-05  1.874579

Columns: ['value']


,value,rolling_30_std
date,,
2023-01-01,0.304717,NaN
2023-01-02,0.027047,NaN
2023-01-03,2.839079,NaN
2023-01-04,3.961909,NaN
2023-01-05,1.874579,NaN
...,...,...
2023-12-27,9.737587,3.336412
2023-12-28,11.340535,3.345714
2023-12-29,12.978562,3.402215
